In [ ]:
"""
台股HFSLS-BIGRU預測系統
包含：HFSLS特徵選擇
不包含：PSO優化（使用固定超參數）
用於消融實驗：驗證HFSLS特徵選擇的效果
"""

import math
import os
import pandas as pd
import numpy as np
from collections import deque
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings
import time
from datetime import timedelta
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ============================================================
# 全局配置
# ============================================================
scaler = MinMaxScaler()
window = 20
loss = "mse"
sample_ratios = [1, 0.9, 0.8]
bins_fs = 3
sub_features = []
ensemble = []
num_models = 3
minloss = 20
s = 0
measure = [[]]
true_pre = [[], []]
best_result_buffer = None
DATA_PATH = './data/'

# ============================================================
# 進度追蹤器
# ============================================================
class ProgressTracker:
    def __init__(self, total_layers, total_stocks):
        self.total_layers = total_layers
        self.total_stocks = total_stocks
        self.stock_start_time = None
        
    def start_stock(self, stock_name, stock_idx):
        self.stock_start_time = time.time()
        print(f"\n{'='*70}")
        print(f" [{stock_idx}/{self.total_stocks}] 處理：{stock_name}")
        print(f"{'='*70}")
    
    def start_layer(self, layer_num):
        print(f"\n{'─'*70}")
        print(f" 第 {layer_num}/{self.total_layers} 層訓練")
        print(f"{'─'*70}")
    
    def end_stock(self, stock_name):
        stock_time = time.time() - self.stock_start_time
        print(f"\n{'='*70}")
        print(f" {stock_name} 完成！耗時 {stock_time/60:.1f} 分鐘")
        print(f"{'='*70}")

tracker = None

# ============================================================
# 核心函數
# ============================================================

def weight(n, k):
    w = np.arange(1, k * n + 1, k)
    return w / w.sum()

def process_data(X):
    que = deque(maxlen=window)
    x = []
    for i in X:
        que.append(i)
        if len(que) == window: x.append(list(que))
    return np.array(x)

def build_fixed_bigru(input_shape, units=64, dropout=0.2):
    model = Sequential([
        Bidirectional(GRU(units, activation='relu', return_sequences=False), input_shape=input_shape),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])
    return model

def train_submodel(x_train, y_train, x_test, y_test, test_indices, dates, stock_name, layer_num):
    global minloss, best_result_buffer
    model = build_fixed_bigru(input_shape=x_train.shape[1:], units=64, dropout=0.2)
    model.fit(x_train, y_train, batch_size=32, epochs=30, validation_split=0.1, verbose=0, shuffle=False)
    
    y_pred = model.predict(x_test, verbose=0)
    mse = mean_squared_error(y_test, y_pred)
    measure[0] = [mse, math.sqrt(mse), mean_absolute_error(y_test, y_pred), mean_absolute_percentage_error(y_test, y_pred), r2_score(y_test, y_pred)]
    
    if mse < minloss:
        y_pred_inv = scaler.inverse_transform(y_pred)
        y_test_inv = scaler.inverse_transform(y_test)
        base_dates = [dates[idx] for idx in test_indices]
        best_result_buffer = pd.DataFrame({'stock_name': stock_name, 'predict_date': [d + pd.Timedelta(days=15) for d in base_dates], 'predicted_close': y_pred_inv[:, 0], 'actual_close': y_test_inv[:, 0]})
        minloss = mse
    return model

def feature_selection(X, y, y_train, N, loss_values, features, w):
    print(f"   特徵選擇 ({len(features)})...", end='')
    F = len(features)
    E = pd.DataFrame({"E_value": np.zeros(F)})
    X_tmp = X.copy()
    for i_f, feat in enumerate(features):
        X_tmp.loc[:, feat] = np.random.permutation(X_tmp.loc[:, feat].values)
        pred = pd.Series(np.zeros(N))
        for i_s, model in enumerate(ensemble):
            x = process_data(X_tmp.loc[:, sub_features[i_s]].values)
            x_train, _, _, _ = train_test_split(x, y, test_size=0.3, shuffle=True)
            pred += pd.Series(model.predict(x_train, verbose=0).squeeze()) * w[i_s]
        loss_feat = (np.squeeze(y_train) - pred.values) ** 2
        E.loc[i_f, "E_value"] = np.mean(loss_feat - loss_values) / (np.std(loss_feat - loss_values) + 1e-7)
        X_tmp.loc[:, feat] = X.loc[:, feat]
    
    E["bins"] = pd.cut(E["E_value"].fillna(0), bins_fs)
    res_feat = []
    for i_b, b in enumerate(sorted(E["bins"].unique(), reverse=True)):
        b_feat = features[E["bins"] == b]
        selected = np.random.choice(b_feat, int(np.ceil(sample_ratios[i_b] * len(b_feat))), replace=False) if len(b_feat) > 0 else []
        res_feat.extend(selected)
    return pd.Index(set(res_feat))

def fit(df, dates, stock_name):
    global measure, minloss, sub_features, ensemble, tracker
    X = df.iloc[:, :-1].apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-7))
    features = X.columns
    y = scaler.fit_transform(df['Close'].values[:-window+1].reshape(-1, 1))
    
    for k in range(num_models):
        if tracker: tracker.start_layer(k+1)
        sub_features.append(features)
        x = process_data(X.loc[:, features].values)
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, shuffle=True)
        model_k = train_submodel(x_train, y_train, x[int(len(y)*0.7):], y[int(len(y)*0.7):], 
                                 range(int(len(y)*0.7), len(y)), dates, stock_name, k+1)
        ensemble.append(model_k)
        if k < num_models - 1: features = feature_selection(X, y, y_train, len(x_train), pd.Series((np.squeeze(y_train) - pd.Series(model_k.predict(x_train, verbose=0).squeeze())) ** 2), features, np.ones(k+1)/(k+1))

def main(data, stock_name):
    global minloss, measure, sub_features, ensemble, best_result_buffer
    minloss, measure, sub_features, ensemble, best_result_buffer = 20, [[]], [], [], None
    df = pd.read_csv(os.path.join(DATA_PATH, data)).fillna(0).sort_index()
    dates = pd.to_datetime(df['Date']) if 'Date' in df.columns else pd.date_range('2021-01-01', periods=len(df), freq='D')
    fit(df.drop(columns=['Date'], errors='ignore'), dates, stock_name)
    if best_result_buffer is not None: best_result_buffer.to_csv(os.path.join(DATA_PATH, 'predicted_hfsls_bigru.csv'), index=False)

if __name__ == '__main__':
    stocks = ['台積電', '長榮', '聯發科', '統一超']
    tracker = ProgressTracker(num_models, len(stocks))
    for idx, s in enumerate(stocks, 1):
        tracker.start_stock(s, idx)
        main(f"{s}_date.csv", s)